# MA-GRPO Full Pipeline Benchmark: Base 7B (Untrained) vs. RLMR Checkpoint-10

Notebook đánh giá **End-to-End Full Pipeline** trên **3 held-out prompts** (`tell-me-a-story-test.jsonl`, mục tiêu **8.000 từ × 8 chương / truyện**) so sánh trực tiếp **2 nhánh** trên **cùng 3 dàn ý Condition D**:
1. **`base_7b`** — `Qwen/Qwen2.5-7B-Instruct` *(chưa train)*
2. **`rlmr_ckpt10`** — `Qwen/Qwen2.5-7B-Instruct` + LoRA **`checkpoint-10`** *(Phase 2 Minimal Orthogonal RLMR, $\gamma = 0.05$)*

**Đầy đủ 3 tầng chấm điểm (GPU A100 80GB, ~12–16 phút tổng cộng, auto-resume trên Google Drive):**
- **Phase 1:** Stage 1 Macro-Only Planning (`Qwen2.5-32B-Instruct-AWQ`, vLLM) + Suite 1 `MacroPlan-Bench` audit
- **Phase 2:** Stage 2 Rolling Writing + ReIO Cumulative LTM + 0-Token Hygiene (`base_7b` vs `rlmr_ckpt10`, native `bf16` + `peft` LoRA)
- **Phase 3:** Suite 2 **`FullStory-Bench`** (`DeBERTa-v3` DOC-NLI + `MiniLM` Continuity + Length/`<think>`/Slop + `32B-AWQ` Judge: **ConStory-Checker 19 lỗi**, **EQ-Bench Longform**, **Blind Pairwise A/B**)
- **Phase 4:** Suite 2A **`ClassicNLG-Bench`** (**BLEU-1..4**, **ROUGE-L**, **Distinct-1..3**, **Entropy-2**, **Sliding-Window PPL**, + **Spearman Correlation**)
- **Phase 5:** Bảng so sánh trực tiếp `Base 7B` vs `RLMR Ckpt-10` (`Δ Delta`) & biểu đồ lưu về Drive để đưa thẳng vào Slide!

In [ ]:
# 1. Mount Google Drive & Sync MA-GRPO Repository
import os
from pathlib import Path
from google.colab import drive

drive.mount('/content/drive')
REPO_DIR = Path('/content/MA-GRPO')
if not REPO_DIR.exists():
    !git clone https://github.com/AIVIETNAM-AIO-TruongDogki/MA_GRPO.git /content/MA-GRPO
else:
    !git -C /content/MA-GRPO pull

%cd /content/MA-GRPO
!nvidia-smi --query-gpu=name,memory.total,memory.free --format=csv

In [ ]:
# 2. Install Dependencies (remove incompatible Colab preinstalled torchao 0.10.0)
!pip uninstall -y -q torchao
!pip install -q vllm peft transformers accelerate sentence-transformers pyyaml pandas matplotlib scipy

In [ ]:
# 3. Experiment Configuration (3 Prompts | Base 7B vs Checkpoint-10)
import json
from pathlib import Path

NUM_PROMPTS = 3
NUM_CHAPTERS = 8
TOTAL_TARGET_WORDS = 8000       # 8 chapters x 1,000 words/chapter
MAX_CHAPTER_TOKENS = 2300       # Matches v2 training ceiling
TEST_DATA_PATH = Path('data/tell-me-a-story-test.jsonl')

PLANNER_MODEL = 'Qwen/Qwen2.5-32B-Instruct-AWQ'
WRITER_BASE_MODEL = 'Qwen/Qwen2.5-7B-Instruct'
JUDGE_MODEL = 'Qwen/Qwen2.5-32B-Instruct-AWQ'
PPL_MODEL = 'Qwen/Qwen2.5-0.5B' # Lightweight causal LM for Sliding-Window Perplexity

# Auto-locate checkpoint-10 (prioritize clean v2 run, fallback to v1 if needed)
CKPT_CANDIDATES = [
    Path('/content/drive/MyDrive/MA-GRPO-Checkpoints/exp02_rlmr_orthogonal_32b_v2/checkpoint-10'),
    Path('/content/drive/MyDrive/MA-GRPO-Checkpoints/exp02_rlmr_orthogonal_32b/checkpoint-10'),
]
CHECKPOINT_DIR = next((p for p in CKPT_CANDIDATES if p.exists()), CKPT_CANDIDATES[0])
assert CHECKPOINT_DIR.exists(), f'Checkpoint not found! Checked: {[str(p) for p in CKPT_CANDIDATES]}'

ARM_BASE = 'base_7b'
ARM_CKPT = 'rlmr_ckpt10'
OUTPUT_DIR = Path('/content/drive/MyDrive/MA-GRPO-Eval/base_vs_ckpt10_3prompts')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

PLANS_PATH = OUTPUT_DIR / 'shared_macro_plans.json'
STORIES_BASE_PATH = OUTPUT_DIR / f'stories_{ARM_BASE}.jsonl'
STORIES_CKPT_PATH = OUTPUT_DIR / f'stories_{ARM_CKPT}.jsonl'
TRANSITIONS_PATH = OUTPUT_DIR / 'coordinator_transitions.jsonl'

print(f'Prompts       : {NUM_PROMPTS} from {TEST_DATA_PATH}')
print(f'Target Length : {TOTAL_TARGET_WORDS}w across {NUM_CHAPTERS} chapters')
print(f'Checkpoint    : {CHECKPOINT_DIR}')
print(f'Output Dir    : {OUTPUT_DIR}')

In [ ]:
# 4. Phase 1 — Generate 3 Shared Condition D Macro Plans (32B-AWQ) + Suite 1 MacroPlan-Bench Audit
import gc
import re
import time
import torch
from src.planning import OutlineAgent, PlanningAgent
from experiments.benchmark.evaluator import (
    strip_length_constraints_from_premise,
    compute_premise_coverage,
    compute_causal_coherence,
    compute_conflict_completeness,
)
from experiments.benchmarks.outline_bench.metrics_pacing import evaluate_concreteness_and_pacing

if PLANS_PATH.exists():
    shared_plans = json.loads(PLANS_PATH.read_text(encoding='utf-8'))
    print(f'[Resume] Loaded {len(shared_plans)} existing shared Condition D macro plans from {PLANS_PATH}')
else:
    raw_rows = [json.loads(line) for line in TEST_DATA_PATH.read_text(encoding='utf-8').splitlines() if line.strip()][:NUM_PROMPTS]
    print(f'[Phase 1] Loading {PLANNER_MODEL} via vLLM to build {len(raw_rows)} Condition D macro plans...', flush=True)
    from vllm import LLM, SamplingParams

    planner_llm = LLM(
        model=PLANNER_MODEL,
        dtype='float16',
        quantization='awq_marlin',
        gpu_memory_utilization=0.82,
        max_model_len=8192,
        trust_remote_code=True,
    )
    planner_tok = planner_llm.get_tokenizer()

    def planner_generate_fn(system_prompt: str, user_prompt: str) -> str:
        prompt_str = planner_tok.apply_chat_template(
            [{'role': 'system', 'content': system_prompt}, {'role': 'user', 'content': user_prompt}],
            tokenize=False,
            add_generation_prompt=True,
        )
        out = planner_llm.generate([prompt_str], SamplingParams(temperature=0.7, top_p=0.9, max_tokens=2048))
        return out[0].outputs[0].text.strip()

    outline_agent = OutlineAgent(llm_generate_fn=planner_generate_fn)
    planning_agent = PlanningAgent(llm_generate_fn=None)

    shared_plans = []
    for idx, row in enumerate(raw_rows):
        pid = str(row.get('example_id') or f'test_{idx:03d}')
        clean_premise = strip_length_constraints_from_premise(str(row.get('inputs', '')))
        t0 = time.time()
        event_graph = outline_agent.create_event_graph(
            premise=clean_premise,
            genre='Creative Fiction',
            num_events=NUM_CHAPTERS,
        )
        blueprint = planning_agent.create_story_plan(
            event_graph=event_graph,
            total_target_words=TOTAL_TARGET_WORDS,
            num_chapters=NUM_CHAPTERS,
            enable_micro_subevents=False,
        )
        s_cov = compute_premise_coverage(clean_premise, blueprint)
        causal_info = compute_causal_coherence(blueprint, nli_scorer=None)
        s_comp = compute_conflict_completeness(blueprint)
        pace_info = evaluate_concreteness_and_pacing(blueprint)
        audit = {
            's_cov': round(float(s_cov), 4),
            's_causal': round(float(causal_info.get('S_causal', 0.0)), 4),
            's_comp': round(float(s_comp), 4),
            'concreteness': round(float(pace_info.get('mean_concreteness', 0.0)), 4),
            'pacing_balance': round(float(pace_info.get('pacing_balance_score', 0.0)), 4),
        }
        shared_plans.append({
            'premise_id': pid,
            'genre': 'Creative Fiction',
            'premise': clean_premise,
            'target_words': TOTAL_TARGET_WORDS,
            'event_graph': event_graph,
            'story_plan': blueprint,
            'outline_audit': audit,
        })
        print(
            f'  [{idx+1}/{len(raw_rows)}] {pid} planned in {time.time()-t0:.1f}s | '
            f'S_cov={audit["s_cov"]:.3f} S_causal={audit["s_causal"]:.3f} '
            f'S_comp={audit["s_comp"]:.3f} Conc={audit["concreteness"]:.3f} '
            f'tokens={event_graph.get("planning_tokens", 0)}',
            flush=True,
        )

    PLANS_PATH.write_text(json.dumps(shared_plans, indent=2, ensure_ascii=False), encoding='utf-8')
    del planner_llm, planner_tok
    gc.collect()
    torch.cuda.empty_cache()
    print(f'[Phase 1 Done] Saved {len(shared_plans)} shared plans -> {PLANS_PATH} & freed 32B VRAM.', flush=True)

In [ ]:
# 5. Phase 2 — Full Rolling 8-Chapter Generation: Base 7B vs RLMR Checkpoint-10 (HF bf16 + PEFT LoRA)
import gc
import time
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel
from src.coordinator import CoordinatorAgent
from src.pipeline import (
    VLLMWritingPolicy,
    extract_cot_and_prose,
    deduplicate_repeated_paragraphs,
)
from src.rewards import count_words

def load_done_ids(path: Path) -> set:
    if not path.exists():
        return set()
    return {json.loads(l)['premise_id'] for l in path.read_text(encoding='utf-8').splitlines() if l.strip()}

base_done = load_done_ids(STORIES_BASE_PATH)
ckpt_done = load_done_ids(STORIES_CKPT_PATH)

if len(base_done) >= len(shared_plans) and len(ckpt_done) >= len(shared_plans):
    print('[Resume] Both base_7b and rlmr_ckpt10 already generated all 3 full stories!')
else:
    torch.backends.cuda.matmul.allow_tf32 = True
    print(f'[Phase 2] Loading {WRITER_BASE_MODEL} in native bf16 + SDPA...', flush=True)
    tok = AutoTokenizer.from_pretrained(WRITER_BASE_MODEL, trust_remote_code=True)
    if tok.pad_token is None:
        tok.pad_token = tok.eos_token
    tok.padding_side = 'left'

    base_model = AutoModelForCausalLM.from_pretrained(
        WRITER_BASE_MODEL,
        torch_dtype=torch.bfloat16,
        device_map='cuda:0',
        attn_implementation='sdpa',
        trust_remote_code=True,
    )
    base_model.eval()

    @torch.inference_mode()
    def hf_chat_generate(model, messages, max_new_tokens: int, temperature: float, top_p: float) -> str:
        prompt_str = tok.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        enc = tok(prompt_str, return_tensors='pt').to(model.device)
        in_len = enc['input_ids'].shape[1]
        do_sample = temperature > 0.05
        gen = model.generate(
            **enc,
            max_new_tokens=max_new_tokens,
            do_sample=do_sample,
            temperature=temperature if do_sample else None,
            top_p=top_p if do_sample else None,
            pad_token_id=tok.pad_token_id,
            eos_token_id=tok.eos_token_id,
            use_cache=True,
        )
        return tok.decode(gen[0][in_len:], skip_special_tokens=True).strip()

    def run_arm(arm_name: str, model, out_path: Path, is_peft: bool):
        done = load_done_ids(out_path)
        policy_helper = VLLMWritingPolicy(use_mock=True)

        def coord_llm_fn(sys_p: str, usr_p: str) -> str:
            # Always run Coordinator with base weights (disable LoRA adapter if PEFT)
            if is_peft:
                with model.disable_adapter():
                    return hf_chat_generate(model, [{'role': 'system', 'content': sys_p}, {'role': 'user', 'content': usr_p}], 640, 0.25, 0.9)
            return hf_chat_generate(model, [{'role': 'system', 'content': sys_p}, {'role': 'user', 'content': usr_p}], 640, 0.25, 0.9)

        coordinator = CoordinatorAgent(llm_generate_fn=coord_llm_fn)

        for idx, item in enumerate(shared_plans):
            pid = item['premise_id']
            if pid in done:
                print(f'  [{arm_name}] Skipping already completed {pid}', flush=True)
                continue
            t_story = time.time()
            blueprint = item['story_plan']
            events = item['event_graph'].get('events', [])
            all_chars = list(dict.fromkeys(ev.get('character', 'Protagonist') for ev in events)) or ['Protagonist']

            chapters_clean = []
            chapters_raw = []
            past_plans = []
            prev_ctx = None

            for ch_plan in blueprint['chapters']:
                ch_idx = ch_plan['chapter_index']
                target_w = int(ch_plan.get('target_words', 1000))
                ctx_state = coordinator.condense_context(
                    next_chapter_index=ch_idx,
                    past_chapters=chapters_clean,
                    past_plans=past_plans,
                    characters=all_chars,
                    previous_context=prev_ctx,
                    next_chapter_plan=ch_plan,
                )
                prev_ctx = ctx_state

                prompt = policy_helper.construct_prompt(ch_plan, ctx_state, enable_cot=True)
                raw_out = hf_chat_generate(model, [{'role': 'user', 'content': prompt}], MAX_CHAPTER_TOKENS, 0.7, 0.9)
                cot_txt, prose_txt = extract_cot_and_prose(raw_out)
                prose_dedup, removed = deduplicate_repeated_paragraphs(prose_txt, min_words=30, min_sentence_words=12)

                # Hygiene retry once at higher temp if dedup trimmed prose below 75% target
                if removed > 0 and count_words(prose_dedup) < int(0.75 * target_w):
                    raw_retry = hf_chat_generate(model, [{'role': 'user', 'content': prompt}], MAX_CHAPTER_TOKENS, 0.85, 0.92)
                    cot_r, prose_r = extract_cot_and_prose(raw_retry)
                    prose_r_dedup, removed_r = deduplicate_repeated_paragraphs(prose_r, min_words=30, min_sentence_words=12)
                    if count_words(prose_r_dedup) > count_words(prose_dedup):
                        raw_out, cot_txt, prose_dedup, removed = raw_retry, cot_r, prose_r_dedup, removed_r

                chapters_raw.append(raw_out)
                chapters_clean.append(prose_dedup)
                past_plans.append(ch_plan)

                # Log transition record for FullStory-Bench Anti-Context Novelty
                with open(TRANSITIONS_PATH, 'a', encoding='utf-8') as tf:
                    tf.write(json.dumps({
                        'arm': arm_name,
                        'premise_id': pid,
                        'chapter_index': ch_idx,
                        'context_summary': ctx_state.get('plot_summary', ''),
                        'prev_chapter_tail': ctx_state.get('recent_tail_excerpt', ''),
                        'is_fallback': False,
                        'dedup_removed': removed,
                        'think_words': len(cot_txt.split()),
                        'prose_words': count_words(prose_dedup),
                    }, ensure_ascii=False) + '\n')

                print(
                    f'    [{arm_name}] {pid} Ch {ch_idx}/{NUM_CHAPTERS}: '
                    f'prose={count_words(prose_dedup)}w (tgt={target_w}w), think={len(cot_txt.split())}w, dedup={removed}',
                    flush=True,
                )

            total_w = sum(count_words(c) for c in chapters_clean)
            rec = {
                'arm': arm_name,
                'premise_id': pid,
                'genre': item['genre'],
                'premise': item['premise'],
                'target_words': TOTAL_TARGET_WORDS,
                'actual_words': total_w,
                'chapters': chapters_clean,
                'chapters_raw_with_think': chapters_raw,
                'story_plan': blueprint,
                'generation_latency_s': round(time.time() - t_story, 2),
            }
            with open(out_path, 'a', encoding='utf-8') as sf:
                sf.write(json.dumps(rec, ensure_ascii=False) + '\n')
            print(f'  [{arm_name}] Saved {pid}: {total_w}w in {rec["generation_latency_s"]}s', flush=True)

    # Arm 1: Base 7B (Untrained)
    print('\n=== Generating Arm 1: Base 7B (Untrained) ===', flush=True)
    run_arm(ARM_BASE, base_model, STORIES_BASE_PATH, is_peft=False)

    # Arm 2: RLMR Checkpoint-10 (PEFT LoRA)
    print(f'\n=== Loading LoRA Adapter: {CHECKPOINT_DIR} ===', flush=True)
    peft_model = PeftModel.from_pretrained(base_model, str(CHECKPOINT_DIR))
    peft_model.eval()
    print('=== Generating Arm 2: RLMR Checkpoint-10 ===', flush=True)
    run_arm(ARM_CKPT, peft_model, STORIES_CKPT_PATH, is_peft=True)

    del peft_model, base_model, tok
    gc.collect()
    torch.cuda.empty_cache()
    print('[Phase 2 Done] All 6 full stories saved & 7B VRAM freed!', flush=True)

In [ ]:
# 6. Phase 3 — Run Suite 2: FullStory-Bench (DOC-NLI + Continuity + ConStory-Checker + EQ-Bench + Blind Pairwise 32B-AWQ)
import subprocess
import sys

FULLSTORY_OUT = OUTPUT_DIR / 'suite2_fullstory_bench'
cmd_fs = [
    sys.executable, '-u', 'scripts/benchmark_fullstory.py',
    '--stories', str(STORIES_BASE_PATH), str(STORIES_CKPT_PATH),
    '--labels', ARM_BASE, ARM_CKPT,
    '--transitions', str(TRANSITIONS_PATH),
    '--judge-backend', 'local_vllm',
    '--judge-model', JUDGE_MODEL,
    '--judge-gpu-mem', '0.82',
    '--judge-max-len', '24576',
    '--pairwise-arms', ARM_BASE, ARM_CKPT,
    '--output-dir', str(FULLSTORY_OUT),
]
print('Running Suite 2 (FullStory-Bench):', ' '.join(cmd_fs), flush=True)
proc = subprocess.Popen(cmd_fs, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
for line in proc.stdout:
    print(line, end='', flush=True)
if proc.wait() != 0:
    raise RuntimeError('FullStory-Bench failed!')

In [ ]:
# 7. Phase 4 — Run Suite 2A: ClassicNLG-Bench (BLEU-1..4, ROUGE-L, Distinct-1..3, Entropy-2, Sliding PPL + Spearman)
CLASSIC_OUT = OUTPUT_DIR / 'suite2a_classic_nlg_bench'
cmd_cnlg = [
    sys.executable, '-u', 'scripts/benchmark_classic_nlg.py',
    '--stories', str(STORIES_BASE_PATH), str(STORIES_CKPT_PATH),
    '--labels', ARM_BASE, ARM_CKPT,
    '--reference-file', str(TEST_DATA_PATH),
    '--reference-fallback', 'premise_and_plan',
    '--ppl-model', PPL_MODEL,
    '--compare-with-fullstory', str(FULLSTORY_OUT / 'fullstory_raw.jsonl'),
    '--output-dir', str(CLASSIC_OUT),
]
print('Running Suite 2A (ClassicNLG-Bench):', ' '.join(cmd_cnlg), flush=True)
proc = subprocess.Popen(cmd_cnlg, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
for line in proc.stdout:
    print(line, end='', flush=True)
if proc.wait() != 0:
    raise RuntimeError('ClassicNLG-Bench failed!')

In [ ]:
# 8. Phase 5 — Side-by-Side Comparison Dashboard (Base 7B vs RLMR Checkpoint-10) + Slide-Ready Plot
import pandas as pd
import matplotlib.pyplot as plt

fs_df = pd.read_csv(FULLSTORY_OUT / 'fullstory_summary.csv').set_index('arm')
cnlg_df = pd.read_csv(CLASSIC_OUT / 'classic_nlg_summary.csv').set_index('arm')
base_fs = fs_df.loc[ARM_BASE].to_dict() if ARM_BASE in fs_df.index else {}
ckpt_fs = fs_df.loc[ARM_CKPT].to_dict() if ARM_CKPT in fs_df.index else {}
base_cl = cnlg_df.loc[ARM_BASE].to_dict() if ARM_BASE in cnlg_df.index else {}
ckpt_cl = cnlg_df.loc[ARM_CKPT].to_dict() if ARM_CKPT in cnlg_df.index else {}

metric_specs = [
    # Suite 2: FullStory-Bench (exact fullstory_summary.csv column names)
    ('FullStory', 'Mean Actual Words (Target=8000)', 'actual_words', 'fs'),
    ('FullStory', 'Length Compliance S_l (%) ↑', 'length_score_sl', 'fs'),
    ('FullStory', 'DOC-NLI Plan Adherence (S_adh) ↑', 'S_adh', 'fs'),
    ('FullStory', 'Inter-Chapter Continuity ↑', 'transition_cosine', 'fs'),
    ('FullStory', 'Anti-Context Novelty ↑', 'anti_context_novelty', 'fs'),
    ('FullStory', 'EQ-Bench Longform (0-100) ↑', 'eqbench_score', 'fs'),
    ('FullStory', 'EQ Meaningful Elaboration ↑', 'eq_meaningful_elaboration', 'fs'),
    ('FullStory', 'ConStory C_weighted ↓', 'C_weighted', 'fs'),
    ('FullStory', 'ConStory Errors / 1k Words ↓', 'constory_errors_per_1k', 'fs'),
    ('FullStory', 'Slop Score / 1k Words ↓', 'slop_score_per_1k', 'fs'),
    ('FullStory', 'Intra-Chapter Redundancy ↓', 'intra_chapter_redundancy', 'fs'),
    ('FullStory', 'Mean <think> Words', 'think_words', 'fs'),
    # Suite 2A: ClassicNLG-Bench (exact classic_nlg_summary.csv column names)
    ('ClassicNLG', 'BLEU-1 ↑', 'BLEU-1', 'cl'),
    ('ClassicNLG', 'BLEU-4 ↑', 'BLEU-4', 'cl'),
    ('ClassicNLG', 'ROUGE-L ↑', 'ROUGE-L', 'cl'),
    ('ClassicNLG', 'Distinct-3 (%) ↑', 'Distinct-3 (%)', 'cl'),
    ('ClassicNLG', 'Entropy-2 ↑', 'Entropy-2', 'cl'),
    ('ClassicNLG', 'Sliding-Window Perplexity ↓', 'Perplexity', 'cl'),
]

def _to_float(v):
    try:
        f = float(v)
        return None if pd.isna(f) else f
    except Exception:
        return None

rows_tbl = []
for suite, label, key, src in metric_specs:
    b_val = _to_float((base_fs if src == 'fs' else base_cl).get(key))
    c_val = _to_float((ckpt_fs if src == 'fs' else ckpt_cl).get(key))
    delta = (c_val - b_val) if (b_val is not None and c_val is not None) else None
    rows_tbl.append({
        'Suite': suite,
        'Metric': label,
        ARM_BASE: round(b_val, 4) if b_val is not None else None,
        ARM_CKPT: round(c_val, 4) if c_val is not None else None,
        'Delta (Ckpt10 - Base)': round(delta, 4) if delta is not None else None,
    })

df = pd.DataFrame(rows_tbl)
print('\n================ BASE 7B vs RLMR CHECKPOINT-10 (3 HELD-OUT PROMPTS) ================')
print(df.to_string(index=False))

pw_path = FULLSTORY_OUT / 'pairwise_verdicts.json'
pairwise = json.loads(pw_path.read_text(encoding='utf-8')) if pw_path.exists() else []
if pairwise:
    wins_base = sum(1 for v in pairwise if v.get('winner') == ARM_BASE)
    wins_ckpt = sum(1 for v in pairwise if v.get('winner') == ARM_CKPT)
    ties = sum(1 for v in pairwise if v.get('winner') == 'tie')
    print(f'\n================ BLIND PAIRWISE 32B JUDGE (SWAPPED ORDER) ================')
    print(f'{ARM_BASE} wins: {wins_base} | {ARM_CKPT} wins: {wins_ckpt} | Ties: {ties}')

corr_path = CLASSIC_OUT / 'classic_vs_fullstory_correlation.json'
corr_data = json.loads(corr_path.read_text(encoding='utf-8')) if corr_path.exists() else {}

# Save unified JSON + CSV to Drive (hand this JSON back to generate final Slide charts!)
unified = {
    'checkpoint': str(CHECKPOINT_DIR),
    'num_prompts': NUM_PROMPTS,
    'target_words': TOTAL_TARGET_WORDS,
    'table': rows_tbl,
    'fullstory_by_arm': {ARM_BASE: base_fs, ARM_CKPT: ckpt_fs},
    'classic_nlg_by_arm': {ARM_BASE: base_cl, ARM_CKPT: ckpt_cl},
    'pairwise_verdicts': pairwise,
    'classic_vs_fullstory_correlation': corr_data,
}
(OUTPUT_DIR / 'unified_comparison_summary.json').write_text(json.dumps(unified, indent=2, ensure_ascii=False), encoding='utf-8')
df.to_csv(OUTPUT_DIR / 'comparison_table.csv', index=False)
print(f'\nSaved unified comparison -> {OUTPUT_DIR / "unified_comparison_summary.json"}')